In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv('../data/processed/sales.csv')
np.random.seed(0)

In [3]:
sales['Date'] = pd.to_datetime(sales['Date'])

In [4]:
print(sales['Date'].dtype)

datetime64[us]


In [ ]:
sales = sales.sort_values('Date').reset_index(drop=True)

In [6]:
# Lag features cho Revenue
for lag in [1, 2, 3, 7, 14, 30]:
    sales[f'revenue_lag_{lag}'] = sales['Revenue'].shift(lag)
# Lag features cho COGS
for lag in [1, 7, 30]:
    sales[f'cogs_lag_{lag}'] = sales['COGS'].shift(lag)

In [8]:
sales.head()

,Date,Revenue,COGS,is_holiday,log_Revenue,log_COGS,revenue_lag_1,revenue_lag_2,revenue_lag_3,revenue_lag_7,revenue_lag_14,revenue_lag_30,cogs_lag_1,cogs_lag_7,cogs_lag_30
0,2012-07-04,5123547.94,3982991.19,0,15.449358,15.197544,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-05,2751773.45,2150580.23,0,14.827757,14.581249,5123547.94,NaN,NaN,NaN,NaN,NaN,3982991.19,NaN,NaN
2,2012-07-06,3054029.42,2517632.84,0,14.931973,14.738830,2751773.45,5123547.94,NaN,NaN,NaN,NaN,2150580.23,NaN,NaN
3,2012-07-07,2667930.94,2108246.62,0,14.796814,14.561368,3054029.42,2751773.45,5123547.94,NaN,NaN,NaN,2517632.84,NaN,NaN
4,2012-07-08,2360851.90,1808622.79,0,14.674534,14.408077,2667930.94,3054029.42,2751773.45,NaN,NaN,NaN,2108246.62,NaN,NaN


In [10]:
# rolling mean
for w in [7, 14, 30]:
    sales[f'rev_roll{w}_mean'] = sales['Revenue'].shift(1).rolling(w).mean()
    if w in [7, 30]: 
        sales[f'rev_roll{w}_std'] = sales['Revenue'].shift(1).rolling(w).std()
        sales[f'rev_roll{w}_max'] = sales['Revenue'].shift(1).rolling(w).max()

In [ ]:
# calendar 
sales['day_of_week'] = sales['Date'].dt.dayofweek 
sales['month'] = sales['Date'].dt.month
sales['day_of_month'] = sales['Date'].dt.day
sales['week_of_year'] = sales['Date'].dt.isocalendar().week.astype(int)
sales['quarter'] = sales['Date'].dt.quarter

sales['is_weekend'] = sales['day_of_week'].isin([5, 6]).astype(int)

fixed_holidays = []
for year in range(2012, 2026):
    fixed_holidays.extend([f'{year}-01-01', f'{year}-04-30', f'{year}-05-01', f'{year}-09-02'])
fixed_holidays = pd.to_datetime(fixed_holidays)

sales['is_holiday'] = sales['Date'].apply(lambda x: 1 if min(abs((x - h).days) for h in fixed_holidays) <= 3 else 0)

# Tet
tet_dates = pd.to_datetime([
    '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08', '2017-01-28', 
    '2018-02-16', '2019-02-05', '2020-01-25', '2021-02-12', '2022-02-01', '2023-01-22', 
    '2024-02-10', '2025-01-29'
])

sales['days_to_tet'] = sales['Date'].apply(lambda x: min(abs((x - t).days) for t in tet_dates)).clip(upper=30)

def check_is_tet(date_val):
    for t in tet_dates:
        diff = (date_val - t).days
        if -3 <= diff <= 7: 
            return 1
    return 0

sales['is_tet'] = sales['Date'].apply(check_is_tet)

# gio to hung vuong
gioto_dates = pd.to_datetime([
    '2012-03-31', '2013-04-19', '2014-04-09', '2015-04-28', '2016-04-16', '2017-04-06', 
    '2018-04-25', '2019-04-14', '2020-04-02', '2021-04-21', '2022-04-10', '2023-04-29', 
    '2024-04-18', '2025-04-07'
])

sales['days_to_gioto'] = sales['Date'].apply(lambda x: min(abs((x - t).days) for t in gioto_dates)).clip(upper=30)

def check_is_gioto(date_val):
    for t in gioto_dates:
        diff = (date_val - t).days
        if -3 <= diff <= 7: 
            return 1
    return 0

sales['is_gioto'] = sales['Date'].apply(check_is_gioto)